# Amazon ML Challenge 2026 — Final Submission Pipeline (Full Scale)

This notebook executes the validated end-to-end Business Entity Resolution pipeline on Google Colab:
- **Blocking**: Phase 3 Multi-Modal Union Blocker + Phase 4 Validated Subword & Digit Keys
- **Candidate Retention**: Tiered Retention capped at 65
- **Pairwise Matcher**: LightGBM with 34 deterministic lexical, address, digit, and interaction features
- **Calibration**: Entity-level train/val split with macro $F_{0.5}$ threshold sweep
- **Inference**: Memory-safe streaming chunked inference (constant RAM footprint; resumable checkpointing)
- **Validation**: Official `utils/validate_submission.py` automated check

### Step 1: Clone Repository & Install Dependencies

In [ ]:
# 1. Clone repository and cd into workspace
!git clone https://github.com/mudassar-quraishi/ML-challenge.git
%cd ML-challenge
!git checkout final-submission-pipeline

# 2. Install dependencies
!pip install -q rapidfuzz lightgbm scikit-learn pandas numpy

### Step 2: Mount Google Drive or Locate Dataset
If your competition dataset is stored in Google Drive, mount it and symlink `dataset/`:

In [ ]:
# Optional: Mount Google Drive if dataset is in Drive
from google.colab import drive
import os

drive.mount('/content/drive')

# Set DATASET_PATH to where your train/ and test/ directories live
# e.g., '/content/drive/MyDrive/AmazonML/dataset'
DATASET_PATH = '/content/drive/MyDrive/AmazonML/dataset'

if os.path.exists(DATASET_PATH):
    if os.path.exists('dataset'):
        !rm -rf dataset
    os.symlink(DATASET_PATH, 'dataset')
    print("Dataset symlinked successfully:")
    !ls -lh dataset
else:
    print(f"Please update DATASET_PATH to the folder containing train/ and test/.")

### Step 3: Run Full Pipeline (Training + Streaming Test Inference)
This will:
1. Fit the LightGBM matcher on a representative 25,000 S1 training sample (~1.5M candidate pairs).
2. Calibrate the decision threshold using macro $F_{0.5}$.
3. Build the candidate index over all test targets (S2 + S3).
4. Stream all 1,732,545 test S1 entities in memory-safe 5,000-record chunks directly to `output/matching_results.tsv` and `output/candidate_pairs.tsv`.
5. Automatically execute `validate_submission.py`.

In [ ]:
!python colab/run_pipeline.py \
    --data-root dataset \
    --output-dir output \
    --train-sample-size 25000 \
    --n-jobs -1

### Step 4: Verify Official Validator Results

In [ ]:
!python utils/validate_submission.py \
    --matching output/matching_results.tsv \
    --candidate output/candidate_pairs.tsv \
    --test-dir dataset/test

### Step 5: Package Final Submission Zip for Leaderboard Upload

In [ ]:
# Create zip containing only the two required TSVs
!zip -j submission_final.zip output/matching_results.tsv output/candidate_pairs.tsv

# Download to local machine or copy to Google Drive
from google.colab import files
files.download('submission_final.zip')

# Also save a copy to Drive
!cp submission_final.zip /content/drive/MyDrive/submission_final.zip
print("Submission zip saved and ready for leaderboard upload!")